### 에이전트(툴)
- llm문제:학습한 이후의 사건이나 사실에 대한 정보(할루시네이션)
이의 해결책으로 나온것
- 에이전트(툴): 위키피디아 등 다른 웹의 소스를 이용
- https://docs.langchain.com/oss/python/integrations/tools

#### llm 질문중 연산이 필요한 경우 사용하는 에이전트

In [5]:
%pip install numexpr

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os
from langchain_openai import ChatOpenAI
from langchain_core.tools import tool
from langchain.agents import create_agent
from langchain_community.agent_toolkits.load_tools import load_tools
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain_community.tools import DuckDuckGoSearchRun 
import requests

In [4]:
@tool
def get_weather(latitude, longitude) -> str:
    """특정 지역 위도,경도의 현재 날씨 정보를 반환합니다."""

    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']


model = ChatOpenAI(model="gpt-4o-mini")

# 에이전트 생성
# 모델과 도구 리스트, 그리고 에이전트의 역할을 정의하는 시스템 프롬프트를 전달합니다.
tools = [get_weather]

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="당신은 유능한 비서입니다. 사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)

# 5. 에이전트 실행
response = agent.invoke(
    {"messages": [{"role": "user", "content": "서울 날씨 어때?"}]}
)
# response = agent.invoke(
#     {"messages": [HumanMessage(content="서울 날씨 어때?")]}
# )
# 마지막 메시지의 내용(content)만 출력
print(response["messages"][-1].content)
# # 결과 출력
for message in response["messages"]:
    message.pretty_print()
    # print( message.content)
    print('=======')

현재 서울의 기온은 -4.3도입니다.
================================ Human Message =================================

서울 날씨 어때?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_8mSZ3NabWMndgf1EHyHBMEjf)
 Call ID: call_8mSZ3NabWMndgf1EHyHBMEjf
  Args:
    latitude: 37.5665
    longitude: 126.978
================================= Tool Message =================================
Name: get_weather

-4.3
================================== Ai Message ==================================

현재 서울의 기온은 -4.3도입니다.


In [3]:
response

{'messages': [HumanMessage(content='서울 날씨 어때?', additional_kwargs={}, response_metadata={}, id='79c0c32f-00b5-477b-a405-8b81e7fd2cda'),
  AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 23, 'prompt_tokens': 85, 'total_tokens': 108, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_29330a9688', 'id': 'chatcmpl-CzXXMAklqxiPADOIcvyf7bsqOCxQA', 'service_tier': 'default', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--019bd3ae-29c0-7160-8211-25e875968624-0', tool_calls=[{'name': 'get_weather', 'args': {'latitude': 37.5665, 'longitude': 126.978}, 'id': 'call_LdBGzgr4bGMrblEvYYdktos5', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 85, 

#### 퀴즈

In [4]:
import pytz
from datetime import datetime

@tool
def get_weather(latitude, longitude) -> str:
    """특정 지역 위도,경도의 현재 날씨 정보를 반환합니다."""

    response = requests.get(f"https://api.open-meteo.com/v1/forecast?latitude={latitude}&longitude={longitude}&current=temperature_2m,wind_speed_10m&hourly=temperature_2m,relative_humidity_2m,wind_speed_10m")
    data = response.json()
    return data['current']['temperature_2m']


@tool
def get_time( city ):
    """구하려는 도시의 IANA Timezone 문자열 (예: Asia/Seoul)"""

    try:
        tz = pytz.timezone( city )
        local_time = datetime.now( tz )
        return local_time.strftime( '시간:%Y-%m-%d %H:%M:%S')
    except Exception as err:
        return "현재 정보로는 시간을 알수 없습니다."
    
model = ChatOpenAI(model="gpt-4o-mini")
tools = [get_time,get_weather]
agent = create_agent(
    model=model,
    tools=tools,
    system_prompt="당신은 유능한 비서입니다. 사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)
response = agent.invoke(
    {"messages": [{"role": "user", "content": "부산의 현재날씨 알려줘"}]}
)
print(response["messages"][-1].content)

for message in response["messages"]:
    message.pretty_print()
    # print( message.content)
    print('=======')

현재 부산의 온도는 4.6도입니다. 추가적인 날씨 정보가 필요하시면 말씀해 주세요.
================================ Human Message =================================

부산의 현재날씨 알려줘
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_zxkETtAeIHKz3UEvsvwSeFe6)
 Call ID: call_zxkETtAeIHKz3UEvsvwSeFe6
  Args:
    latitude: 35.1796
    longitude: 129.0756
================================= Tool Message =================================
Name: get_weather

4.6
================================== Ai Message ==================================

현재 부산의 온도는 4.6도입니다. 추가적인 날씨 정보가 필요하시면 말씀해 주세요.


In [ ]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = load_tools(["wikipedia", "llm-math"], llm=llm)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)
question = "그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?"

response = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)
# print(response)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?
================================== Ai Message ==================================
Tool Calls:
  wikipedia (call_kRs5nsZkhwPCGGTdC2TNfZa7)
 Call ID: call_kRs5nsZkhwPCGGTdC2TNfZa7
  Args:
    query: Queen band leader birth year
================================= Tool Message =================================
Name: wikipedia

Page: Queen Camilla
Summary: Camilla (born Camilla Rosemary Shand, later Parker Bowles, 17 July 1947) is Queen of the United Kingdom and 14 other Commonwealth realms as the wife of King Charles III.
Camilla was raised in East Sussex and South Kensington in England and educated in England, Switzerland and France. In 1973, she married British Army officer Andrew Parker Bowles; they divorced in 1995. Camilla and Charles were romantically involved periodically, both before and during each of their first marriages. Their relationship was highly publicised in th

In [17]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

@tool
def get_weather(location: str) -> str:
    """특정 지역의 현재 날씨 정보를 반환합니다."""
    # 실제로는 API를 호출하겠지만, 예시를 위해 고정값을 반환합니다.
    return f"{location}의 날씨는 맑음, 기온은 25도입니다."

tools = load_tools(["wikipedia", "llm-math"], llm=llm)
tools.append( get_weather )

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)
# question = "그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?"
question = "서울 날씨 어때"
# response = agent.invoke(
#     {"messages": [{"role": "user", "content": question}]}
# )
input_messages = [HumanMessage(content="서울 날씨 어때?")]
response = agent.invoke(   {"messages":input_messages})

for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

서울 날씨 어때?
================================== Ai Message ==================================
Tool Calls:
  get_weather (call_XYaUOEX9bMEIk6wtDl7ck1Nr)
 Call ID: call_XYaUOEX9bMEIk6wtDl7ck1Nr
  Args:
    location: 서울
================================= Tool Message =================================
Name: get_weather

서울의 날씨는 맑음, 기온은 25도입니다.
================================== Ai Message ==================================

서울의 날씨는 맑고, 기온은 25도입니다.


In [1]:
from langchain_community.agent_toolkits.load_tools import get_all_tool_names
get_all_tool_names()

['sleep',
 'wolfram-alpha',
 'google-search',
 'google-search-results-json',
 'searx-search-results-json',
 'bing-search',
 'metaphor-search',
 'ddg-search',
 'google-books',
 'google-lens',
 'google-serper',
 'google-scholar',
 'google-finance',
 'google-trends',
 'google-jobs',
 'google-serper-results-json',
 'searchapi',
 'searchapi-results-json',
 'serpapi',
 'dalle-image-generator',
 'twilio',
 'searx-search',
 'merriam-webster',
 'wikipedia',
 'arxiv',
 'golden-query',
 'pubmed',
 'human',
 'awslambda',
 'stackexchange',
 'sceneXplain',
 'graphql',
 'openweathermap-api',
 'dataforseo-api-search',
 'dataforseo-api-search-json',
 'eleven_labs_text2speech',
 'google_cloud_texttospeech',
 'read_file',
 'reddit_search',
 'news-api',
 'tmdb-api',
 'podcast-api',
 'memorize',
 'llm-math',
 'open-meteo-api',
 'requests',
 'requests_get',
 'requests_post',
 'requests_patch',
 'requests_put',
 'requests_delete',
 'terminal']

In [4]:
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

tools = load_tools(["ddg-search", "llm-math"], llm=llm)

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)
question = "그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?"

response = agent.invoke(
    {"messages": [{"role": "user", "content": question}]}
)
# print(response)
for message in response["messages"]:
    message.pretty_print()

================================ Human Message =================================

그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?
================================== Ai Message ==================================
Tool Calls:
  duckduckgo_search (call_rKaEnqM3Yml8Zi8pDccUvQc1)
 Call ID: call_rKaEnqM3Yml8Zi8pDccUvQc1
  Args:
    query: Queen band leader birth year
================================= Tool Message =================================
Name: duckduckgo_search

1 day ago - Queen + Paul Rodgers performed ... ninetieth birthday , and again promote awareness of the HIV/AIDS pandemic. The first Queen + Paul Rodgers album, titled The Cosmos Rocks, was released in Europe on 12 September 2008 and in the US on 28 October 2008. The band again toured ... 14 hours ago - Freddie Mercury (born Farrokh Bulsara; 5 September 1946 – 24 November 1991) was a British singer and songwriter who achieved global fame as the lead vocalist and pianist of the rock band Queen. Regarded as one of the greatest singers in the h

In [8]:
search = DuckDuckGoSearchRun()
search.invoke("Obama's first name?")

"Obama promoted inclusion for LGBT Americans, becoming the first sitting U. S . president to publicly support same-sex marriage. англ. Barack Hussein Obama II[ 1 ]. Отец. Барак Хуссейн Обама — старший. Obama ' s First Retrospective Job Approval Rating Is 63%. Институт Гэллапа (англ.). Obama ’ s father, also named Barack Hussein Obama , grew up in a small village in Nyanza Province, Kenya, as a member of the Luo ethnicity. He won a scholarship to study economics at the... Obama Sr. married three times and had children by four women. With his first wife Kezia, Obama Sr. had a son and a daughter, Roy and Auma, before Barack was born. As President Obama has said, the change we seek will take longer than one term or one presidency.Welcome to the Office of Barack and Michelle Obama . We Love You Back."

- DuckDuckGoSearch: 실시간 정보, 최신 뉴스, 현재 사건에 적합
- Wikipedia:정적 지식, 인물, 역사, 개념 설명에 적합

In [ ]:
import uuid
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
mytools = load_tools(["wikipedia", "llm-math"], llm=llm)
print('tool', mytools)
search = DuckDuckGoSearchRun(description="반드시 실시간 정보, 최신 뉴스, 현재 사건, 오늘 기준 정보가 필요할 때만 사용")
tools = [search] + mytools

agent = create_agent(
    model=llm,
    tools=tools,
    system_prompt="사용자의 질문에 도구를 사용하여 정확하게 답하세요."
)

# question = "그룹 퀸의 리더가 태어난 해는? 2026년 현재는 몇 살?"
question = "최신뉴스 top5 알려줄래"

input_messages = [HumanMessage(content=question)]
response = agent.invoke(   {"messages":input_messages})

for message in response["messages"]:
    message.pretty_print()

tool [WikipediaQueryRun(api_wrapper=WikipediaAPIWrapper(wiki_client=<module 'wikipedia' from 'c:\\Python310\\lib\\site-packages\\wikipedia\\__init__.py'>, top_k_results=3, lang='en', load_all_available_meta=False, doc_content_chars_max=4000)), Tool(name='Calculator', description='Useful for when you need to answer questions about math.', func=<bound method Chain.run of LLMMathChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, template='Translate a math problem into a expression that can be executed using Python\'s numexpr library. Use the output of running this code to answer the question.\n\nQuestion: ${{Question with math problem.}}\n```text\n${{single line mathematical expression that solves the problem}}\n```\n...numexpr.evaluate(text)...\n```output\n${{Output of running the code}}\n```\nAnswer: ${{Answer}}\n\nBegin.\n\nQuestion: What is 37593 * 67?\n```text\n37593 * 67\n```\n...numexpr.ev

In [16]:
type(mytools[0] )

langchain_community.tools.wikipedia.tool.WikipediaQueryRun

In [16]:
print(response["messages"][-1].content)

최신 뉴스 상위 5개는 다음과 같습니다:

1. **서울의 강추위**: 14일 서울의 아침 최저기온이 -9도까지 떨어지는 등 강추위가 계속될 것으로 보입니다. 아침 최저기온은 -15∼-2도로 예년보다 낮고, 낮 최고기온은 -2∼11도로 예년에 비해 조금 높을 것으로 예상됩니다.

2. **정류장 떠난 퇴근길 시민들**: 퇴근길 시민들이 정류장에서 버스를 기다리고 있지만, 버스는 아직 차고지에 있는 상황이 보도되었습니다.

3. **미국 내 총격 사건**: 미니애폴리스에서 연방 요원이 관련된 또 다른 총격 사건이 발생했습니다.

4. **이란의 시위와 처벌**: 이란 당국은 전국적인 시위에 대한 빠른 재판과 처형이 예정되어 있다고 밝혔으며, 미국이나 이스라엘이 개입할 경우 보복하겠다고 경고했습니다.

5. **국제 뉴스**: 이란의 보안 관계자는 반정부 시위 진압 과정에서 2,000명이 사망했다고 전했습니다.

이 뉴스들은 현재 진행 중인 사건들과 관련된 중요한 정보들입니다.
